**RAG as a sequential pipeline**

I outlined the complete lifecycle of an advanced RAG (Retrieval-Augmented Generation) Pipeline  

1. Preprocessing

2. Index Construction

3. LLM-Augmented Retrieval

4. Evaluation


**Evaluating the final output**

I outlined the fundamental pillars to evaluate the results

4.1 Retrieval Quality

4.2 Generation Quality

4.3 End-to-End QA Accuracy

**2. INDEX CONSTRUCTION**

***2.1 Load Dataset***

I imported the RAG1 output 

In [7]:
import pickle

with open("/kaggle/input/datasets/saridacharoenngampit/import-preprocessing/documents.pkl", "rb") as f:
    documents = pickle.load(f)

print(type(documents))
print(type(documents[0]))

<class 'list'>
<class 'dict'>


***2.2 Chunking***

I tried to chunk the text; however, the method is not suitable for the data since one question is preferably to be considered as one chunk

In [8]:
def chunk_text(text, chunk_size=600):
    return [text[i:i+chunk_size]
        for i in range(0, len(text), chunk_size)]

chunks = chunk_text(documents[0]["text"])

print("Total chunks:", len(chunks))
print(chunks[0][:600])

Total chunks: 2

QUESTION: a 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. she states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. she otherwise feels well and is followed by a doctor for her pregnancy. her temperature is 97.7°f (36.5°c), blood pressure is 122/77 mmhg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. which of the following is the best treatment for this patient?
OPTIO


In [9]:
question_texts = [doc["text"] for doc in documents]
print(len(question_texts))


10178


Notes: the total questions are 10178 which could consume a significant amount of time to embed

***2.3 Embedding***

I imported torch, passed device CUDA to the SentenceTransformer constructor, and formatted embedding as a batch

In [10]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Runing on: {device}")


Runing on: cpu


In [ ]:
!pip install ipywidgets

from sentence_transformers import SentenceTransformer

embedding_model =  SentenceTransformer(model_name_or_path="all-MiniLM-L6-v2")

**Notes:**  UNEXPECTED is a known benign mismatch that commonly appears after Transformers-version changes. Since position_ids is typically a non-trainable buffer, ignoring it usually has no effect on inference or fine-tuning. Source: https://discuss.huggingface.co/t/embeddings-position-ids-unexpected-warning-started-showing/173102

In [ ]:
embeddings = embedding_model.encode(question_texts,batch_size=1267,normalize_embeddings=True,
show_progress_bar=True)


***2.3.1 FAISS Embedding***

I converted Python list of embeddings into a dense NumPy array, passed the number of dimensions, and embedded the vectors

In [19]:
!pip install -q faiss-cpu

In [20]:
import faiss
import numpy as np

In [21]:
embeddings = np.array(embeddings).astype("float32")

faiss_index = faiss.IndexFlatIP(embeddings.shape[1])
faiss_index.add(embeddings)


print(faiss_index.ntotal)

10178


In [33]:
faiss.write_index(faiss_index, "faiss_index.bin")

***2.3.2 BM25 Embedding***

I converted all characters to lowercase, splited sentences into arrays of individual words, and built a statistical index

In [22]:
!pip install -q rank-bm25

In [23]:
from rank_bm25 import BM25Okapi

In [25]:
tokenized_docs = [doc["text"].lower().split()
    for doc in documents]
bm25 = BM25Okapi(tokenized_docs)

print(tokenized_docs[:2])

[['question:', 'a', '23-year-old', 'pregnant', 'woman', 'at', '22', 'weeks', 'gestation', 'presents', 'with', 'burning', 'upon', 'urination.', 'she', 'states', 'it', 'started', '1', 'day', 'ago', 'and', 'has', 'been', 'worsening', 'despite', 'drinking', 'more', 'water', 'and', 'taking', 'cranberry', 'extract.', 'she', 'otherwise', 'feels', 'well', 'and', 'is', 'followed', 'by', 'a', 'doctor', 'for', 'her', 'pregnancy.', 'her', 'temperature', 'is', '97.7°f', '(36.5°c),', 'blood', 'pressure', 'is', '122/77', 'mmhg,', 'pulse', 'is', '80/min,', 'respirations', 'are', '19/min,', 'and', 'oxygen', 'saturation', 'is', '98%', 'on', 'room', 'air.', 'physical', 'exam', 'is', 'notable', 'for', 'an', 'absence', 'of', 'costovertebral', 'angle', 'tenderness', 'and', 'a', 'gravid', 'uterus.', 'which', 'of', 'the', 'following', 'is', 'the', 'best', 'treatment', 'for', 'this', 'patient?', 'options:a.', 'ampicillin', 'b.', 'ceftriaxone', 'c.', 'ciprofloxacin', 'd.', 'doxycycline', 'e.', 'nitrofurantoin',

In [ ]:
import pickle
with open("tokenized_docs.pkl", "wb") as f:
    pickle.dump(tokenized_docs, f)